<p align="center">
  <img src="../assets/prodinno_logo.png" alt="Prodinno" width="200">
</p>

<h4 align="center">Bonus · Deep Dive</h4>
<h1 align="center">A Neural Network, From Scratch</h1>
<p align="center"><i>2 inputs, 3 hidden neurons, 2 outputs -- every forward pass and every gradient, written out in raw numpy.</i></p>

---

## Why this notebook exists

Logistic regression, as built in Session 1, is $\sigma(w^Tx + b)$ -- **one neuron**. It draws
exactly one straight (or hyperplane) decision boundary, because that is all a single linear
combination followed by a sigmoid can express. A **neural network** is nothing mystical on top
of that: it is several of these neurons arranged in layers, each layer's output feeding the
next layer's input. Stacking neurons like this is what lets the model bend its decision
boundary into a curve -- something no amount of tuning can make a single logistic regression
neuron do.

This notebook builds the smallest network that can show that off end to end, with **zero**
machine-learning libraries -- just numpy:

$$\underbrace{2}_{\text{inputs}} \rightarrow \underbrace{3}_{\text{hidden neurons}} \rightarrow \underbrace{2}_{\text{outputs}}$$

We'll derive the forward pass, derive backpropagation by hand, watch the network's decision
boundary curve into shape over training, and compare three different activation functions
head to head on identical data.

In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.datasets import make_moons

RANDOM_STATE = 3
BLUE, RED, GREEN, ORANGE, PURPLE = "#4C72B0", "#C44E52", "#55A868", "#DD8452", "#8172B2"

# Two interleaving crescents - a classic dataset NO straight line can separate.
# This is exactly the case where a hidden layer earns its keep over plain logistic regression.
X, y = make_moons(n_samples=240, noise=0.22, random_state=RANDOM_STATE)

# One-hot encode the 2 classes - this is why the network has 2 OUTPUT neurons, not 1:
# output neuron k learns to predict P(class = k).
Y = np.zeros((len(y), 2))
Y[np.arange(len(y)), y] = 1.0

fig_data = go.Figure()
fig_data.add_trace(go.Scatter(x=X[y == 0, 0], y=X[y == 0, 1], mode="markers", marker=dict(color=BLUE, size=7), name="class 0"))
fig_data.add_trace(go.Scatter(x=X[y == 1, 0], y=X[y == 1, 1], mode="markers", marker=dict(color=RED, size=7), name="class 1"))
fig_data.update_layout(title="Two moons - not linearly separable", height=420, width=560, xaxis_title="x1", yaxis_title="x2")
fig_data.show()

**Why this dataset:** no single straight line can separate the two crescents -- try
drawing one. Logistic regression's linear decision boundary is fundamentally incapable of
solving this. A network with even one hidden layer can, because the hidden layer first
**warps the space** the data lives in before the output layer draws its (still linear, but now
much more useful) boundary through the warped version.

## 1. Activation Functions

Every hidden neuron computes a **weighted sum** of its inputs, then passes that sum through a
nonlinear **activation function**. Without that nonlinearity, stacking layers would be
pointless -- a linear function of a linear function is still just linear, so a "deep" network
of purely linear layers would collapse into a single linear model, no more powerful than
logistic regression. The activation is what gives depth its power. Three common choices:

$$\text{sigmoid}(z) = \frac{1}{1+e^{-z}} \qquad \tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}} \qquad \text{ReLU}(z) = \max(0, z)$$

And their derivatives -- needed for backpropagation in Section 3:

$$\text{sigmoid}'(z) = \sigma(z)\big(1-\sigma(z)\big) \qquad \tanh'(z) = 1 - \tanh^2(z) \qquad \text{ReLU}'(z) = \begin{cases} 1 & z > 0 \\ 0 & z \leq 0 \end{cases}$$

In [2]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_deriv(z):
    s = sigmoid(z)
    return s * (1 - s)

def tanh_act(z):
    return np.tanh(z)

def tanh_deriv(z):
    return 1 - np.tanh(z) ** 2

def relu(z):
    return np.maximum(0, z)

def relu_deriv(z):
    return (z > 0).astype(float)

def softmax(z):
    # subtract the row-max first - purely for numerical stability, does not change the result
    z = z - z.max(axis=1, keepdims=True)
    ez = np.exp(z)
    return ez / ez.sum(axis=1, keepdims=True)

ACTS = {"sigmoid": (sigmoid, sigmoid_deriv), "tanh": (tanh_act, tanh_deriv), "relu": (relu, relu_deriv)}

z_range = np.linspace(-6, 6, 300)
fig_act = make_subplots(rows=1, cols=2, subplot_titles=("Activation functions", "Their derivatives"))
for name, color in [("sigmoid", BLUE), ("tanh", GREEN), ("relu", RED)]:
    act_fn, deriv_fn = ACTS[name]
    fig_act.add_trace(go.Scatter(x=z_range, y=act_fn(z_range), mode="lines", name=name, line=dict(color=color)), row=1, col=1)
    fig_act.add_trace(go.Scatter(x=z_range, y=deriv_fn(z_range), mode="lines", name=name, line=dict(color=color), showlegend=False), row=1, col=2)
fig_act.update_xaxes(title_text="z")
fig_act.update_layout(height=420, width=950, title="sigmoid vs. tanh vs. ReLU")
fig_act.show()

**What to notice:** sigmoid's derivative maxes out at just $0.25$ and shrinks toward
zero for large $|z|$ -- this is the **vanishing gradient** problem, where a saturated sigmoid
neuron barely updates at all. tanh is steeper (derivative maxes at $1$) and zero-centered,
usually training a little faster. ReLU's derivative is either exactly $0$ or exactly $1$ --
cheap to compute and never saturates for $z>0$, but a ReLU neuron that drifts into $z\leq 0$
for every training example ("dies") stops learning entirely, since its gradient is then
permanently zero. We'll see this exact failure mode later in Section 5.

## 2. The Network Architecture (2 → 3 → 2)

$$
\underbrace{z_1 = xW_1 + b_1}_{\text{hidden pre-activation}} \quad\rightarrow\quad
\underbrace{a_1 = f(z_1)}_{\text{hidden activation}} \quad\rightarrow\quad
\underbrace{z_2 = a_1 W_2 + b_2}_{\text{output pre-activation}} \quad\rightarrow\quad
\underbrace{a_2 = \text{softmax}(z_2)}_{\text{predicted class probabilities}}
$$

with shapes $W_1 \in \mathbb{R}^{2\times3}$, $b_1 \in \mathbb{R}^{3}$,
$W_2 \in \mathbb{R}^{3\times2}$, $b_2 \in \mathbb{R}^{2}$ -- 6 + 3 + 6 + 2 = **17 learnable
numbers** in total, for a network this small. Below is that architecture drawn out, with a
random, untrained set of weights -- blue edges are positive weights, red are negative, and
edge thickness is proportional to magnitude.

In [3]:
def network_diagram(W1, W2, title):
    in_y, hid_y, out_y = [0.7, 0.3], [0.85, 0.5, 0.15], [0.65, 0.35]
    in_x, hid_x, out_x = 0.0, 1.0, 2.0
    max_w = max(np.abs(W1).max(), np.abs(W2).max())

    fig = go.Figure()
    for i in range(2):
        for h in range(3):
            w = W1[i, h]
            fig.add_trace(go.Scatter(
                x=[in_x, hid_x], y=[in_y[i], hid_y[h]], mode="lines",
                line=dict(color=BLUE if w >= 0 else RED, width=1 + 6 * abs(w) / max_w),
                opacity=0.7, showlegend=False, hoverinfo="text", text=f"W1[{i},{h}] = {w:.2f}",
            ))
    for h in range(3):
        for o in range(2):
            w = W2[h, o]
            fig.add_trace(go.Scatter(
                x=[hid_x, out_x], y=[hid_y[h], out_y[o]], mode="lines",
                line=dict(color=BLUE if w >= 0 else RED, width=1 + 6 * abs(w) / max_w),
                opacity=0.7, showlegend=False, hoverinfo="text", text=f"W2[{h},{o}] = {w:.2f}",
            ))

    fig.add_trace(go.Scatter(x=[in_x] * 2, y=in_y, mode="markers+text", marker=dict(size=34, color=PURPLE),
                              text=["x1", "x2"], textposition="middle center", textfont=dict(color="white", size=13), showlegend=False, hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=[hid_x] * 3, y=hid_y, mode="markers+text", marker=dict(size=34, color=GREEN),
                              text=["h1", "h2", "h3"], textposition="middle center", textfont=dict(color="white", size=13), showlegend=False, hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=[out_x] * 2, y=out_y, mode="markers+text", marker=dict(size=34, color=ORANGE),
                              text=["y0", "y1"], textposition="middle center", textfont=dict(color="white", size=13), showlegend=False, hoverinfo="skip"))

    fig.update_layout(title=title, height=420, width=650, xaxis=dict(visible=False, range=[-0.3, 2.3]), yaxis=dict(visible=False, range=[0, 1]), plot_bgcolor="white")
    return fig


def init_params(seed, n_in=2, n_hidden=3, n_out=2, scale=1.0):
    r = np.random.RandomState(seed)
    W1 = r.randn(n_in, n_hidden) * scale
    b1 = np.zeros(n_hidden)
    W2 = r.randn(n_hidden, n_out) * scale
    b2 = np.zeros(n_out)
    return W1, b1, W2, b2


W1_init, b1_init, W2_init, b2_init = init_params(seed=1)
network_diagram(W1_init, W2_init, "Untrained network - random weights").show()

## 3. Backpropagation, Derived

Training means adjusting $W_1, b_1, W_2, b_2$ to minimize the **categorical cross-entropy**
loss $J = -\frac{1}{n}\sum_i \sum_k y_{ik}\log(a_{2,ik})$. Backpropagation is just the chain
rule, applied layer by layer, back to front:

**Output layer.** When softmax is paired with cross-entropy, their gradients combine into a
famously clean result -- the messy derivative of softmax and the messy derivative of $\log$
cancel almost entirely, leaving just:

$$\frac{\partial J}{\partial z_2} = a_2 - y \qquad\text{(predicted probabilities minus the true one-hot label)}$$

From there, ordinary chain rule for a linear layer:

$$\frac{\partial J}{\partial W_2} = a_1^T \, \frac{\partial J}{\partial z_2} \qquad \frac{\partial J}{\partial b_2} = \sum_i \frac{\partial J}{\partial z_2}$$

**Hidden layer.** Push the error signal backward through $W_2$, then through the hidden
activation's own derivative $f'$:

$$\frac{\partial J}{\partial a_1} = \frac{\partial J}{\partial z_2} W_2^T \qquad \frac{\partial J}{\partial z_1} = \frac{\partial J}{\partial a_1} \odot f'(z_1) \qquad \frac{\partial J}{\partial W_1} = x^T \frac{\partial J}{\partial z_1} \qquad \frac{\partial J}{\partial b_1} = \sum_i \frac{\partial J}{\partial z_1}$$

That $f'(z_1)$ term is exactly why the choice of activation function matters for *training*,
not just for what the network can represent -- it directly scales how much gradient reaches
$W_1$ at every single step.

In [4]:
def forward(inputs, W1, b1, W2, b2, act_fn):
    z1 = inputs @ W1 + b1
    a1 = act_fn(z1)
    z2 = a1 @ W2 + b2
    a2 = softmax(z2)
    return z1, a1, z2, a2


def loss_fn(a2, targets, eps=1e-9):
    return -np.mean(np.sum(targets * np.log(a2 + eps), axis=1))


def accuracy(a2, labels):
    return (a2.argmax(axis=1) == labels).mean()


def train(act_name, lr, n_epochs, seed=1, scale=1.0, checkpoint_epochs=None):
    '''Full training loop: forward pass, the backprop equations from Section 3, then a gradient step.'''
    act_fn, act_deriv = ACTS[act_name]
    W1, b1, W2, b2 = init_params(seed, scale=scale)
    n = len(X)
    loss_hist, acc_hist = [], []
    checkpoints = {}
    if checkpoint_epochs and 0 in checkpoint_epochs:
        checkpoints[0] = (W1.copy(), b1.copy(), W2.copy(), b2.copy())

    for epoch in range(1, n_epochs + 1):
        z1, a1, z2, a2 = forward(X, W1, b1, W2, b2, act_fn)
        loss_hist.append(loss_fn(a2, Y))
        acc_hist.append(accuracy(a2, y))

        dz2 = (a2 - Y) / n                  # the clean softmax + cross-entropy gradient
        dW2 = a1.T @ dz2
        db2 = dz2.sum(axis=0)
        da1 = dz2 @ W2.T
        dz1 = da1 * act_deriv(z1)           # chain rule through the hidden activation
        dW1 = X.T @ dz1
        db1 = dz1.sum(axis=0)

        W1 -= lr * dW1
        b1 -= lr * db1
        W2 -= lr * dW2
        b2 -= lr * db2

        if checkpoint_epochs and epoch in checkpoint_epochs:
            checkpoints[epoch] = (W1.copy(), b1.copy(), W2.copy(), b2.copy())

    return dict(W1=W1, b1=b1, W2=W2, b2=b2, loss=loss_hist, acc=acc_hist, checkpoints=checkpoints)

## 4. Watching It Learn -- Animated Decision Boundary

We train with `tanh` hidden activations and snapshot the network's weights at increasingly
spaced-out points during training. At each snapshot we ask the network for its predicted
$P(\text{class }1)$ across a fine grid covering the input space, and color the plane by that
probability -- this is the network's **decision boundary**, made visible.

In [5]:
CHECKPOINT_EPOCHS = sorted({0, 2, 5, 10, 20, 40, 80, 150, 250, 400, 600, 900, 1300, 1800, 2400, 3000, 4000})
result_tanh = train("tanh", lr=1.0, n_epochs=4000, seed=1, checkpoint_epochs=CHECKPOINT_EPOCHS)
print(f"final loss = {result_tanh['loss'][-1]:.4f},  final accuracy = {result_tanh['acc'][-1]:.4f}")

pad = 0.6
gx = np.linspace(X[:, 0].min() - pad, X[:, 0].max() + pad, 100)
gy = np.linspace(X[:, 1].min() - pad, X[:, 1].max() + pad, 100)
GX, GY = np.meshgrid(gx, gy)
grid_pts = np.column_stack([GX.ravel(), GY.ravel()])
act_fn_tanh = ACTS["tanh"][0]


def prob_grid(W1, b1, W2, b2, act_fn):
    _, _, _, a2 = forward(grid_pts, W1, b1, W2, b2, act_fn)
    return a2[:, 1].reshape(GX.shape)


def status_at(W1, b1, W2, b2, act_fn):
    _, _, _, a2 = forward(X, W1, b1, W2, b2, act_fn)
    return loss_fn(a2, Y), accuracy(a2, y)


frames = []
for ep in CHECKPOINT_EPOCHS:
    W1c, b1c, W2c, b2c = result_tanh["checkpoints"][ep]
    Z = prob_grid(W1c, b1c, W2c, b2c, act_fn_tanh)
    l, a = status_at(W1c, b1c, W2c, b2c, act_fn_tanh)
    frames.append(go.Frame(
        name=str(ep), data=[go.Heatmap(z=Z, x=gx, y=gy)], traces=[0],
        layout=go.Layout(annotations=[dict(text=f"epoch {ep} \u00b7 loss={l:.3f} \u00b7 acc={a:.3f}", x=0.02, y=1.08, xref="paper", yref="paper", showarrow=False, font=dict(size=13))]),
    ))

W1_0, b1_0, W2_0, b2_0 = result_tanh["checkpoints"][0]
Z0 = prob_grid(W1_0, b1_0, W2_0, b2_0, act_fn_tanh)

fig_boundary = go.Figure()
fig_boundary.add_trace(go.Heatmap(z=Z0, x=gx, y=gy, colorscale="RdBu_r", zmid=0.5, showscale=True, colorbar=dict(title="P(class 1)")))
fig_boundary.add_trace(go.Scatter(x=X[y == 0, 0], y=X[y == 0, 1], mode="markers", marker=dict(color=BLUE, size=7, line=dict(color="white", width=0.5)), name="class 0"))
fig_boundary.add_trace(go.Scatter(x=X[y == 1, 0], y=X[y == 1, 1], mode="markers", marker=dict(color=RED, size=7, line=dict(color="white", width=0.5)), name="class 1"))

fig_boundary.frames = frames
fig_boundary.update_layout(
    title="Decision boundary learning to curve around the two moons",
    height=520, width=650,
    annotations=[dict(text="epoch 0", x=0.02, y=1.08, xref="paper", yref="paper", showarrow=False, font=dict(size=13))],
    updatemenus=[dict(type="buttons", showactive=False, y=1.15, x=1.0, xanchor="right", buttons=[
        dict(label="Play", method="animate", args=[None, dict(frame=dict(duration=350, redraw=True), fromcurrent=True)]),
        dict(label="Pause", method="animate", args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate")]),
    ])],
    sliders=[dict(steps=[dict(method="animate", args=[[str(ep)], dict(mode="immediate", frame=dict(duration=0, redraw=True))], label=str(ep)) for ep in CHECKPOINT_EPOCHS], x=0.1, len=0.85)],
    xaxis_title="x1", yaxis_title="x2",
)
fig_boundary.show()

final loss = 0.1004,  final accuracy = 0.9583


**Reading the animation:** at epoch 0 the untrained network's boundary is essentially
arbitrary. Within the first few dozen epochs it already curves to follow the general shape of
the moons, and by a few hundred epochs it has wrapped itself around both crescents almost
perfectly. **This exact curve is impossible for plain logistic regression** -- it is only
possible because the hidden layer first bends the input space, and it is only *learnable*
because backpropagation can assign credit (and gradient) all the way back through that bend to
every one of the 17 weights and biases.

## 5. Does the Choice of Activation Function Matter?

Same architecture, same data, same learning rate, same random seed for the initial
weights -- the *only* thing that changes below is the hidden layer's activation function.

In [6]:
results = {}
for act_name, color in [("sigmoid", BLUE), ("tanh", GREEN), ("relu", RED)]:
    r = train(act_name, lr=1.0, n_epochs=4000, seed=1)
    Z = prob_grid(r["W1"], r["b1"], r["W2"], r["b2"], ACTS[act_name][0])
    results[act_name] = dict(Z=Z, loss=r["loss"], acc=r["acc"], color=color)
    print(f"{act_name:<8} final loss={r['loss'][-1]:.4f}  final accuracy={r['acc'][-1]:.4f}")

fig_compare = make_subplots(rows=1, cols=4, subplot_titles=["sigmoid", "tanh", "ReLU", "loss vs. epoch"])
for col, act_name in enumerate(["sigmoid", "tanh", "relu"], start=1):
    fig_compare.add_trace(go.Heatmap(z=results[act_name]["Z"], x=gx, y=gy, colorscale="RdBu_r", zmid=0.5, showscale=False), row=1, col=col)
    fig_compare.add_trace(go.Scatter(x=X[y == 0, 0], y=X[y == 0, 1], mode="markers", marker=dict(color=BLUE, size=4), showlegend=False), row=1, col=col)
    fig_compare.add_trace(go.Scatter(x=X[y == 1, 0], y=X[y == 1, 1], mode="markers", marker=dict(color=RED, size=4), showlegend=False), row=1, col=col)

for act_name in ["sigmoid", "tanh", "relu"]:
    r = results[act_name]
    fig_compare.add_trace(go.Scatter(y=r["loss"], mode="lines", name=act_name, line=dict(color=r["color"])), row=1, col=4)

fig_compare.update_yaxes(title_text="loss", row=1, col=4)
fig_compare.update_xaxes(title_text="epoch", row=1, col=4)
fig_compare.update_layout(height=380, width=1400, title="Same architecture, same data, same learning rate - only the hidden activation differs")
fig_compare.show()

sigmoid  final loss=0.1193  final accuracy=0.9583


tanh     final loss=0.1004  final accuracy=0.9583


relu     final loss=0.2887  final accuracy=0.8708


**Reading the comparison:** sigmoid and tanh both wrap a clean boundary around the two
crescents and converge smoothly. ReLU, on this particular run, gets **stuck** -- its loss curve
plateaus early and its boundary stays mostly linear, far short of the other two. With only 3
hidden units, it's easy for one or more ReLU neurons to drift into $z \leq 0$ for every single
training point early on; once that happens $\text{ReLU}'(z)=0$ everywhere for that neuron, no
gradient ever reaches its incoming weights again, and the network is left trying to solve a
curved problem with fewer *effective* neurons than it started with. This is the real-world
**"dying ReLU"** problem, and it is precisely why wider networks (dozens or hundreds of hidden
units, not 3), better weight initialization schemes, or variants like **Leaky ReLU**
($\max(0.01z, z)$, which never fully zeroes its gradient) are standard practice once networks
grow beyond toy size.

## 6. The Learned Network

Here is the same architecture diagram from Section 2, now filled in with the weights the
`tanh` network in Section 4 actually learned.

In [7]:
network_diagram(result_tanh["W1"], result_tanh["W2"], "Trained network - learned weights (tanh, 4000 epochs)").show()

**Reading the diagram:** compare this to the random, untrained version in Section 2 --
the weights are no longer arbitrary noise. Some connections have grown strong (thick edges,
either color) because that particular input-to-hidden or hidden-to-output pathway turned out
to carry real predictive signal for separating the two moons; others have shrunk toward zero
because they didn't. Every one of these 17 numbers was moved to exactly where it is by nothing
more than the handful of matrix multiplications and the chain-rule equations from Section 3,
repeated 4000 times. That -- a system with no explicit "if this curve, then that label" logic
anywhere in it, arriving at a genuinely curved, accurate decision boundary purely by
repeatedly nudging 17 numbers downhill -- is the real beauty of gradient descent and
backpropagation: the same two mechanisms from Session 1's single neuron, scaled up, are the
entire engine behind every deep neural network in production today.

## Summary

- Built a 2 → 3 → 2 neural network from raw numpy: forward pass, softmax output,
  categorical cross-entropy loss, and the full backpropagation gradient derivation --
  including the clean $a_2 - y$ simplification that softmax + cross-entropy produces together.
- Visualized the network as a weighted graph, before and after training, to make the model's
  17 learnable numbers tangible rather than abstract.
- Animated the decision boundary curving into shape over 4000 training epochs on a
  two-moons dataset that plain logistic regression cannot solve -- direct, visual proof of
  what a hidden layer buys you.
- Compared sigmoid, tanh, and ReLU hidden activations head to head on identical data and
  initialization, and watched ReLU visibly get stuck via the real "dying ReLU" failure mode --
  motivating why production networks use far more than 3 hidden units, careful
  initialization, and activations like Leaky ReLU.